In [10]:
from langgraph.graph import StateGraph,START, END , MessagesState
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI 
from dotenv import load_dotenv
from typing import TypedDict
from langchain_core.messages.utils import trim_messages 
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

In [4]:
def token_counter(messages):
    total_chars = 0
    
    for msg in messages:
        total_chars += len(msg.content)
    
    # Approx: 1 token ≈ 4 chars
    return total_chars // 4

In [7]:
def chat_node(state : MessagesState):
    messages = trim_messages(state['messages'],
                            strategy = 'last',
                            token_counter = token_counter,
                            max_tokens = 200,
                            )
    print("Current token Count : ", token_counter(messages=messages))
    for message in messages:
        print(message.content)
    response = llm.invoke(messages)
    return {'messages': [response]}


In [11]:
checkpointer = InMemorySaver()
graph = StateGraph(MessagesState)
graph.add_node("Chat_Node",chat_node)
graph.add_edge(START, "Chat_Node")
graph.add_edge("Chat_Node", END)
workflow = graph.compile(checkpointer=checkpointer)

In [14]:
config = {"configurable":{"thread_id": "thread_1"}}

In [15]:
workflow.invoke({"messages" : [{"role": "user", "content": "What is the capital of France?"}]},config=config)

Current token Count :  7
What is the capital of France?


{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='e94d7bb7-eb93-47c3-b4ee-e9cbb39a03a0'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-ce302d26-4522-4778-b0e6-62e5fe840bb6-0', usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}})]}

In [16]:
workflow.invoke({"messages" : [{"role": "user", "content": "What is my name?"}]},config=config)

Current token Count :  20
What is the capital of France?
The capital of France is **Paris**.
What is my name?


{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='e94d7bb7-eb93-47c3-b4ee-e9cbb39a03a0'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-ce302d26-4522-4778-b0e6-62e5fe840bb6-0', usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='f38ddbaa-f0d7-4006-94c8-b3ee79a033b2'),
  AIMessage(content="As an AI, I don't know your name. I don't have access to personal information about you.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-d2230c7e-1ed2-4140-ad0a-af29cad8d6f7-0', usage_metadata={'input

In [18]:
workflow.invoke({"messages" : [{"role": "user", "content": "my name is John Doe. tell me about trimming in Context Window in short"}]},config=config)

Current token Count :  59
What is the capital of France?
The capital of France is **Paris**.
What is my name?
As an AI, I don't know your name. I don't have access to personal information about you.
my name is John Doe. tell me about trimming in Context Window in short


{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='e94d7bb7-eb93-47c3-b4ee-e9cbb39a03a0'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-ce302d26-4522-4778-b0e6-62e5fe840bb6-0', usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='f38ddbaa-f0d7-4006-94c8-b3ee79a033b2'),
  AIMessage(content="As an AI, I don't know your name. I don't have access to personal information about you.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-d2230c7e-1ed2-4140-ad0a-af29cad8d6f7-0', usage_metadata={'input

In [19]:
workflow.invoke({"messages" : [{"role": "user", "content": "What is my name?"}]},config=config)

Current token Count :  182
my name is John Doe. tell me about trimming in Context Window in short
Hello John Doe!

"Trimming in the Context Window" refers to how AI models manage their limited "short-term memory" during a conversation.

1.  **Context Window:** This is the fixed-size buffer where the AI keeps the current prompt and a history of previous turns. It has a maximum token limit (like a word count).
2.  **Trimming:** When the conversation gets too long and exceeds this limit, the system *trims* (removes) the oldest messages from the history to make space for new ones.

**In short:** It's the AI "forgetting" earlier parts of a long conversation to make room for current input, ensuring it stays within its processing capacity.
What is my name?


{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='e94d7bb7-eb93-47c3-b4ee-e9cbb39a03a0'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-ce302d26-4522-4778-b0e6-62e5fe840bb6-0', usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='f38ddbaa-f0d7-4006-94c8-b3ee79a033b2'),
  AIMessage(content="As an AI, I don't know your name. I don't have access to personal information about you.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-d2230c7e-1ed2-4140-ad0a-af29cad8d6f7-0', usage_metadata={'input

In [20]:
workflow.invoke({"messages" : [{"role": "user", "content": "tell me about summarization in context window"}]},config=config)

Current token Count :  200
my name is John Doe. tell me about trimming in Context Window in short
Hello John Doe!

"Trimming in the Context Window" refers to how AI models manage their limited "short-term memory" during a conversation.

1.  **Context Window:** This is the fixed-size buffer where the AI keeps the current prompt and a history of previous turns. It has a maximum token limit (like a word count).
2.  **Trimming:** When the conversation gets too long and exceeds this limit, the system *trims* (removes) the oldest messages from the history to make space for new ones.

**In short:** It's the AI "forgetting" earlier parts of a long conversation to make room for current input, ensuring it stays within its processing capacity.
What is my name?
Your name is **John Doe**.
tell me about summarization in context window


{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='e94d7bb7-eb93-47c3-b4ee-e9cbb39a03a0'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-ce302d26-4522-4778-b0e6-62e5fe840bb6-0', usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='f38ddbaa-f0d7-4006-94c8-b3ee79a033b2'),
  AIMessage(content="As an AI, I don't know your name. I don't have access to personal information about you.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-d2230c7e-1ed2-4140-ad0a-af29cad8d6f7-0', usage_metadata={'input

In [21]:
workflow.invoke({"messages" : [{"role": "user", "content": "What is my name?"}]},config=config)

Current token Count :  4
What is my name?


{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='e94d7bb7-eb93-47c3-b4ee-e9cbb39a03a0'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-ce302d26-4522-4778-b0e6-62e5fe840bb6-0', usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='f38ddbaa-f0d7-4006-94c8-b3ee79a033b2'),
  AIMessage(content="As an AI, I don't know your name. I don't have access to personal information about you.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-d2230c7e-1ed2-4140-ad0a-af29cad8d6f7-0', usage_metadata={'input